In [42]:
import pandas as pd
import numpy as np

# Same 24h preprocessed features you already built for regression
X_train_linear = pd.read_csv("../../../data/processed/24_hour_prediction/X_train_linear.csv")
X_test_linear  = pd.read_csv("../../../data/processed/24_hour_prediction/X_test_linear.csv")
X_train_tree   = pd.read_csv("../../../data/processed/24_hour_prediction/X_train_tree.csv")
X_test_tree    = pd.read_csv("../../../data/processed/24_hour_prediction/X_test_tree.csv")

y_train_reg = pd.read_csv("../../../data/processed/24_hour_prediction/y_train.csv").squeeze()
y_test_reg  = pd.read_csv("../../../data/processed/24_hour_prediction/y_test.csv").squeeze()

# Binarize: Safe (<=35.4 ug/m3, EPA Good+Moderate) vs Unsafe (>35.4, USG+Unhealthy+VeryUnhealthy+Hazardous)
THRESHOLD = 35.4

y_train = (y_train_reg > THRESHOLD).astype(int)  # 1 = Unsafe, 0 = Safe
y_test  = (y_test_reg > THRESHOLD).astype(int)

print("Train class balance:")
print(y_train.value_counts(normalize=True))
print("\nTest class balance:")
print(y_test.value_counts(normalize=True))

Train class balance:
target
0    0.709788
1    0.290212
Name: proportion, dtype: float64

Test class balance:
target
0    0.640388
1    0.359612
Name: proportion, dtype: float64


In [63]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

log_reg = LogisticRegression()
log_reg.fit(X_train_linear, y_train)

# Predict probabilities for the positive class
y_prob = log_reg.predict_proba(X_test_linear)[:, 1]

# Set your desired decision threshold
threshold = 0.28

# Convert probabilities to class predictions
y_pred = (y_prob >= threshold).astype(int)

print(f"Threshold : {threshold:.2f}")
print(f"Accuracy  : {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision : {precision_score(y_test, y_pred):.4f}")
print(f"Recall    : {recall_score(y_test, y_pred):.4f}")
print(f"F1        : {f1_score(y_test, y_pred):.4f}")
print(f"Confusion matrix:\n{confusion_matrix(y_test, y_pred)}")

Threshold : 0.28
Accuracy  : 0.9100
Precision : 0.8501
Recall    : 0.9102
F1        : 0.8792
Confusion matrix:
[[2040  202]
 [ 113 1146]]


d:\projects\pm25-forecasting\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [94]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import numpy as np

log_reg_l1 = LogisticRegression(
    penalty="l1",
    solver="liblinear",
    C=0.01,
    max_iter=2000
)

log_reg_l1.fit(X_train_linear, y_train)

# Predict probabilities
y_prob = log_reg_l1.predict_proba(X_test_linear)[:, 1]

# Set decision threshold
threshold = 0.3

# Convert probabilities to class labels
y_pred = (y_prob >= threshold).astype(int)

n_nonzero = np.sum(log_reg_l1.coef_ != 0)

print(f"Threshold : {threshold:.2f}")
print(f"Accuracy  : {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision : {precision_score(y_test, y_pred):.4f}")
print(f"Recall    : {recall_score(y_test, y_pred):.4f}")
print(f"F1        : {f1_score(y_test, y_pred):.4f}")
print(f"Nonzero coefficients: {n_nonzero} / {X_train_linear.shape[1]}")
print(f"Confusion matrix:\n{confusion_matrix(y_test, y_pred)}")

d:\projects\pm25-forecasting\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
d:\projects\pm25-forecasting\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


Threshold : 0.30
Accuracy  : 0.9135
Precision : 0.8504
Recall    : 0.9214
F1        : 0.8845
Nonzero coefficients: 20 / 129
Confusion matrix:
[[2038  204]
 [  99 1160]]


In [97]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

rf_clf = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    min_samples_leaf=5,
    n_jobs=-1,
    random_state=42
)

rf_clf.fit(X_train_tree, y_train)

# Predict probabilities for the positive class
y_prob = rf_clf.predict_proba(X_test_tree)[:, 1]

# Set your desired decision threshold
threshold = 0.3

# Convert probabilities to class labels
y_pred = (y_prob >= threshold).astype(int)

print(f"Threshold: {threshold:.2f}")
print(f"Accuracy : {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred):.4f}")
print(f"F1       : {f1_score(y_test, y_pred):.4f}")
print(f"Confusion matrix:\n{confusion_matrix(y_test, y_pred)}")

Threshold: 0.30
Accuracy : 0.9080
Precision: 0.8515
Recall   : 0.9015
F1       : 0.8758
Confusion matrix:
[[2044  198]
 [ 124 1135]]


In [159]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

xgb_clf = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.01,
    n_jobs=-1,
    scale_pos_weight=2.5,
    random_state=42
)

xgb_clf.fit(X_train_tree, y_train)

# Predict probabilities for the positive class
y_prob = xgb_clf.predict_proba(X_test_tree)[:, 1]

# Set your desired decision threshold
threshold = 0.44

# Convert probabilities to class labels
y_pred = (y_prob >= threshold).astype(int)

print(f"Threshold: {threshold:.2f}")
print(f"Accuracy : {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred):.4f}")
print(f"F1       : {f1_score(y_test, y_pred):.4f}")
print(f"Confusion matrix:\n{confusion_matrix(y_test, y_pred)}")

Threshold: 0.44
Accuracy : 0.9097
Precision: 0.8500
Recall   : 0.9095
F1       : 0.8787
Confusion matrix:
[[2040  202]
 [ 114 1145]]


In [187]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import numpy as np

torch.manual_seed(42)
np.random.seed(42)

# Rebuild raw PM2.5 windows, same as your regression LSTM, but target is now binary
raw_data = pd.read_csv("../../../data/raw/kathmandu_full_raw_2023_2024.csv")
raw_data["time"] = pd.to_datetime(raw_data["time"])
raw_data = raw_data.sort_values("time").reset_index(drop=True)

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
pm25_scaled = scaler.fit_transform(raw_data[["pm2_5"]])

def create_windows_classification(df_scaled, raw_values, window=12, horizon=24, threshold=35.4):
    X, y = [], []
    for i in range(len(df_scaled) - window - horizon + 1):
        X.append(df_scaled[i : i+window])
        target_val = raw_values[i + window + horizon - 1]
        y.append(1 if target_val > threshold else 0)
    return np.array(X), np.array(y)

X_lstm, y_lstm = create_windows_classification(
    pm25_scaled, raw_data["pm2_5"].values, window=12, horizon=24, threshold=35.4
)

split = int(np.ceil(0.8 * len(X_lstm)))
X_train_lstm, X_test_lstm = X_lstm[:split], X_lstm[split:]
y_train_lstm, y_test_lstm = y_lstm[:split], y_lstm[split:]

X_train_t = torch.tensor(X_train_lstm, dtype=torch.float32)
y_train_t = torch.tensor(y_train_lstm, dtype=torch.float32).unsqueeze(1)
X_test_t  = torch.tensor(X_test_lstm,  dtype=torch.float32)

loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=64, shuffle=False)

class LSTMClassifier(nn.Module):
    def __init__(self, hidden=100, layers=2):
        super().__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=hidden, num_layers=layers, batch_first=True)
        self.linear = nn.Linear(hidden, 1)  # outputs a logit, not a probability

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.linear(out[:, -1, :])

model = LSTMClassifier()
criterion = nn.BCEWithLogitsLoss()  # combines sigmoid + binary cross-entropy
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(5):
    model.train()
    for xb, yb in loader:
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1}/10 - Loss: {loss.item():.4f}")

Epoch 1/10 - Loss: 0.1551
Epoch 2/10 - Loss: 0.0236
Epoch 3/10 - Loss: 0.0191
Epoch 4/10 - Loss: 0.0212
Epoch 5/10 - Loss: 0.0230


In [188]:

model.eval()
with torch.no_grad():
    logits = model(X_test_t)
    probs = torch.sigmoid(logits).numpy().flatten()
    y_pred = (probs > 0.39).astype(int)

print(f"Accuracy : {accuracy_score(y_test_lstm, y_pred):.4f}")
print(f"Precision: {precision_score(y_test_lstm, y_pred):.4f}")
print(f"Recall   : {recall_score(y_test_lstm, y_pred):.4f}")
print(f"F1       : {f1_score(y_test_lstm, y_pred):.4f}")
print(f"Confusion matrix:\n{confusion_matrix(y_test_lstm, y_pred)}")

Accuracy : 0.9123
Precision: 0.8500
Recall   : 0.9182
F1       : 0.8828
Confusion matrix:
[[2038  204]
 [ 103 1156]]


In [189]:
pm2_5 = ["scaler__pm2_5"] + [f"scaler__pm2_5_lag_{i}" for i in range(1, 13)]
cols = pm2_5 + ["remainder__hour_sin", "remainder__hour_cos", "remainder__dow_sin", "remainder__dow_cos", "remainder__month_sin", "remainder__month_cos"]

In [191]:
X_train = X_train_linear[cols]
X_test = X_test_linear[cols]

In [209]:
log_reg = LogisticRegression()
log_reg.fit(X_train, y_train)

# Predict probabilities for the positive class
y_prob = log_reg.predict_proba(X_test)[:, 1]

# Set your desired decision threshold
threshold = 0.2307

# Convert probabilities to class predictions
y_pred = (y_prob >= threshold).astype(int)

print(f"Threshold : {threshold:.2f}")
print(f"Accuracy  : {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision : {precision_score(y_test, y_pred):.4f}")
print(f"Recall    : {recall_score(y_test, y_pred):.4f}")
print(f"F1        : {f1_score(y_test, y_pred):.4f}")
print(f"Confusion matrix:\n{confusion_matrix(y_test, y_pred)}")

Threshold : 0.23
Accuracy  : 0.9137
Precision : 0.8500
Recall    : 0.9230
F1        : 0.8850
Confusion matrix:
[[2037  205]
 [  97 1162]]
